In [1]:
from torchvision import models
import torch.nn as nn
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd
import json

In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[0]
sys.path.append(str(ROOT))

from Utils.project_utils import *
import Utils.constants as c
from Utils.transforms import *
from Utils.MushroomDataset import *
from Utils.dataLoaders import *

In [3]:
PROJECT_ROOT = get_project_root()
CLASSES_DIR = PROJECT_ROOT / "Classes"
DATA_DIR = PROJECT_ROOT / "Data"

device = get_device()

Using device: mps


In [4]:
SPLIT_DIR = DATA_DIR / "splits"

train = pd.read_csv(SPLIT_DIR / "train.csv")
val = pd.read_csv(SPLIT_DIR / "val.csv")
test = pd.read_csv(SPLIT_DIR / "test.csv")

with open(SPLIT_DIR / "class_to_idx.json", "r") as f:
    train_class_to_idx = json.load(f)

#### 2nd Model - ResNet50

##### Build Data Loaders

In [5]:
train_tfms_mild = get_train_tfms_mild()
train_tfms_strong = get_train_tfms_strong()
val_tfms = get_val_tfms()

In [6]:
CLASSES = sorted(train["class"].unique().tolist())

In [ ]:
loaders = make_loaders(
    train,
    val,
    test,
    train_tfms_mild,
    train_tfms_strong,
    val_tfms,
    train_class_to_idx,
    CLASSES,
    c.BATCH_SIZE,
    c.NUM_WORKERS,
    device,
)

loaders

{'train_loader_mild': <torch.utils.data.dataloader.DataLoader at 0x3294294d0>,
 'train_loader_strong': <torch.utils.data.dataloader.DataLoader at 0x14e9ed290>,
 'val_loader': <torch.utils.data.dataloader.DataLoader at 0x32555a490>,
 'test_loader': <torch.utils.data.dataloader.DataLoader at 0x329429310>}

In [ ]:
train_loader_imb = loaders["train_loader_strong"]
train_loader = loaders["train_loader_strong"]
val_loader = loaders["val_loader"]
test_loader = loaders["test_loader"]

In [ ]:
EPOCHS = 5
LR = 1e-4
NUM_CLASSES = len(CLASSES)

##### Train